In [97]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [98]:
# adding file path location and mounting it
file_path = "/content/drive/MyDrive/TrainingNames.txt"

with open(file_path, "r") as f:    names = f.read().splitlines()

names = ["^" + name + "$" for name in names]

print(names[:5])

['^Radha Soni$', '^Manav Nayar$', '^Manav Sawant$', '^Tarun Raina$', '^Harish Gaikwad$']


In [99]:
# finding number of unique charcter
chars = sorted(list(set("".join(names))))
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

vocab_size = len(chars)
print("Vocab size:", vocab_size)

Vocab size: 51


In [100]:
import torch

def encode(name):
    return [stoi[c] for c in name]

def decode(seq):
    return "".join([itos[i] for i in seq])

In [101]:
#displaying function for each model parameters
def print_model_info(model, model_name, lr=None, epochs=None):
    print(f"\nModel: {model_name}")

    # Total parameters
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable Parameters - {total_params}")

    # Trying to extract layer details
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Embedding):
            print(f"Embedding Dimension  - {module.embedding_dim}")

        if isinstance(module, torch.nn.RNN):
            print(f"Hidden Size          - {module.hidden_size}")
            print(f"Number of Layers     - {module.num_layers}")

        if isinstance(module, torch.nn.LSTM):
            print(f"Hidden Size          - {module.hidden_size}")
            print(f"Number of Layers     - {module.num_layers}")

        if isinstance(module, torch.nn.Dropout):
            print(f"Dropout              - {module.p}")

    if lr:
        print(f"Learning Rate        - {lr}")
    if epochs:
        print(f"Epochs               - {epochs}")

In [102]:
#RNN model
import torch.nn as nn

class CharRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.fc(out)
        return out

In [103]:
#hyperparametrs initialization
hidden_size = 128
lr = 0.001
epochs = 50

In [104]:
#RNN training
model = CharRNN(vocab_size, hidden_size)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(epochs):
    total_loss = 0

    for name in names:
        seq = encode(name)
        x = torch.tensor(seq[:-1]).unsqueeze(0)
        y = torch.tensor(seq[1:]).unsqueeze(0)

        pred = model(x)
        loss = loss_fn(pred.view(-1, vocab_size), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 2195.2791
Epoch 2, Loss: 1720.1793
Epoch 3, Loss: 1488.8099
Epoch 4, Loss: 1338.7074
Epoch 5, Loss: 1230.9709
Epoch 6, Loss: 1155.1812
Epoch 7, Loss: 1099.4189
Epoch 8, Loss: 1054.0751
Epoch 9, Loss: 1023.2071
Epoch 10, Loss: 993.7590
Epoch 11, Loss: 964.7069
Epoch 12, Loss: 940.6452
Epoch 13, Loss: 922.5575
Epoch 14, Loss: 905.0605
Epoch 15, Loss: 898.9500
Epoch 16, Loss: 884.7725
Epoch 17, Loss: 867.6937
Epoch 18, Loss: 855.7462
Epoch 19, Loss: 849.0151
Epoch 20, Loss: 840.0399
Epoch 21, Loss: 836.4037
Epoch 22, Loss: 834.8160
Epoch 23, Loss: 818.0429
Epoch 24, Loss: 819.1835
Epoch 25, Loss: 802.0961
Epoch 26, Loss: 820.0772
Epoch 27, Loss: 807.2610
Epoch 28, Loss: 793.1890
Epoch 29, Loss: 791.4576
Epoch 30, Loss: 791.6532
Epoch 31, Loss: 775.9200
Epoch 32, Loss: 779.8482
Epoch 33, Loss: 779.7340
Epoch 34, Loss: 773.0895
Epoch 35, Loss: 762.8922
Epoch 36, Loss: 759.2934
Epoch 37, Loss: 762.9553
Epoch 38, Loss: 760.6069
Epoch 39, Loss: 759.0387
Epoch 40, Loss: 755.8221


In [105]:
#Name Generation
import random

def generate(model, max_len=19):
    model.eval()

    result = ["^"]   # start token

    for _ in range(max_len):
        x = torch.tensor([stoi[c] for c in result]).unsqueeze(0)

        out = model(x)
        probs = torch.softmax(out[0, -1], dim=0)

        #  Using sampling
        temperature = 0.8
        probs = torch.softmax(out[0, -1] / temperature, dim=0)

        ch = itos[torch.multinomial(probs, 1).item()]

        if ch == "$":
            break

        result.append(ch)

    return "".join(result[1:])

# Generating samples
for _ in range(10):
    print(generate(model))



Sahil Upadhyay
Nilam Goswami
Anisha Kaur
Mayani Acharya
Nitin Gupta
Alok Kaul
Krish Bajaj
Anisha Gaikwad
Sejal Palande
Jitesh Ojha


In [106]:
print_model_info(model, "Vanilla RNN", lr=0.001, epochs=50)


Model: Vanilla RNN
Trainable Parameters - 46131
Embedding Dimension  - 128
Hidden Size          - 128
Number of Layers     - 1
Learning Rate        - 0.001
Epochs               - 50


In [107]:
#Novelity Rate
clean_names = [name[1:-1] for name in names]

generated = [generate(model) for _ in range(200)]

novel = [name for name in generated if name not in clean_names]

novelty_rate = len(novel) / len(generated)
print("Novelty Rate:", novelty_rate)

Novelty Rate: 0.65


In [108]:
#Diversity Rate
diversity = len(set(generated)) / len(generated)
print("Diversity:", diversity)

Diversity: 0.945


In [109]:
#Bilstm implementation
import torch
import torch.nn as nn

class BLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_size=128):
        super().__init__()

        self.hidden_size = hidden_size

        self.embed = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_size,
            batch_first=True,
            bidirectional=True
        )

        #forward only
        self.fc = nn.Linear(hidden_size, vocab_size)

        self.drop = nn.Dropout(0.3)

    def forward(self, x):
        x = self.drop(self.embed(x))
        out, _ = self.lstm(x)

        # take only forward direction
        out = out[:, :, :self.hidden_size]

        return self.fc(out)

    # hidden state init
    def init_hidden(self):
        return (
            torch.zeros(2, 1, self.hidden_size),
            torch.zeros(2, 1, self.hidden_size)
        )

    #step-by-step forward
    def forward_step(self, x, hidden):
        emb = self.embed(x)
        out, hidden = self.lstm(emb, hidden)

        out = out[:, :, :self.hidden_size]
        out = self.fc(out[:, -1])

        return out, hidden



# TRAINING
model = BLSTM(vocab_size)

optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
loss_fn = nn.CrossEntropyLoss()

epochs = 60

for epoch in range(epochs):
    total_loss = 0

    for name in names:
        seq = encode(name)

        x = torch.tensor(seq[:-1]).unsqueeze(0)
        y = torch.tensor(seq[1:]).unsqueeze(0)

        pred = model(x)
        loss = loss_fn(pred.view(-1, vocab_size), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")



# GENERATION


def generate(model, max_len=20, temperature=0.75):
    model.eval()

    hidden = model.init_hidden()
    x = torch.tensor([[stoi['^']]])

    result = []

    for _ in range(max_len):
        logits, hidden = model.forward_step(x, hidden)

        probs = torch.softmax(logits / temperature, dim=-1)
        idx = torch.multinomial(probs, 1).item()

        ch = itos[idx]

        if ch == '$':
            break

        result.append(ch)
        x = torch.tensor([[idx]])

    return "".join(result)



# SAMPLE OUTPUT


print("\nGenerated Names:")
samples = [generate(model) for _ in range(10)]
for name in samples:
    print(name)



# EVALUATION


clean_names = [name[1:-1] for name in names]

generated = [generate(model) for _ in range(200)]

novel = [n for n in generated if n not in clean_names]
novelty = len(novel) / len(generated)

diversity = len(set(generated)) / len(generated)

print("\nEvaluation:")
print("Novelty:", novelty)
print("Diversity:", diversity)



# PARAMETERS


params = sum(p.numel() for p in model.parameters())
print("\nParameters:", params)

Epoch 10, Loss: 1026.3256
Epoch 20, Loss: 896.9494
Epoch 30, Loss: 833.3972
Epoch 40, Loss: 794.3132
Epoch 50, Loss: 759.9099
Epoch 60, Loss: 737.1593

Generated Names:
Falak Choudhury
Ritu Jadeja
Gunjan Moshi
Rakhi Pawar
Vijay Gupta
Aahan Desai
Mahima Trivedi
Chandan Dutta
Lalita Dalmia
Prasad Kapur

Evaluation:
Novelty: 0.355
Diversity: 0.905

Parameters: 208499


In [110]:
print_model_info(model, "BLSTM", lr=0.002, epochs=60)


Model: BLSTM
Trainable Parameters - 208499
Embedding Dimension  - 64
Hidden Size          - 128
Number of Layers     - 1
Dropout              - 0.3
Learning Rate        - 0.002
Epochs               - 60


In [111]:
#RNN with attention
import torch
import torch.nn as nn

class RNNWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_size=128):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_size,
            batch_first=True
        )

        # attention
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=4,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        emb = self.dropout(self.embed(x))
        H, _ = self.lstm(emb)

        T = H.size(1)

        # causal mask
        mask = torch.triu(torch.ones(T, T), diagonal=1).bool()

        attn_out, _ = self.attn(H, H, H, attn_mask=mask)

        out = H + attn_out   # residual
        return self.fc(out)

In [113]:

model = RNNWithAttention(vocab_size)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0015)
loss_fn = nn.CrossEntropyLoss()

epochs = 60

for epoch in range(epochs):
    total_loss = 0

    for name in names:
        seq = encode(name)

        x = torch.tensor(seq[:-1]).unsqueeze(0)
        y = torch.tensor(seq[1:]).unsqueeze(0)

        pred = model(x)
        loss = loss_fn(pred.view(-1, vocab_size), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 10, Loss: 1022.2223
Epoch 20, Loss: 884.7980
Epoch 30, Loss: 827.8324
Epoch 40, Loss: 782.1722
Epoch 50, Loss: 760.2547
Epoch 60, Loss: 733.4615


In [114]:
#generating names
def generate(model, max_len=25, temperature=0.75):
    model.eval()

    x = torch.tensor([[stoi['^']]])
    result = []
    has_space = False

    for _ in range(max_len):
        out = model(x)
        logits = out[0, -1]

        probs = torch.softmax(logits / temperature, dim=0)
        idx = torch.multinomial(probs, 1).item()

        ch = itos[idx]

        # preventing early stop
        if ch == '$' and len(result) < 5:
            continue

        # surname control
        if ch == ' ':
            if has_space:
                continue
            has_space = True

        if ch == '$' and not has_space:
            continue

        if ch == '$':
            break

        result.append(ch)

        x = torch.cat([x, torch.tensor([[idx]])], dim=1)

    return "".join(result)

In [115]:
print("\nGenerated Names:")
samples = [generate(model) for _ in range(10)]
for name in samples:
    print(name)


Generated Names:
Vijay Asthana
Shreya Vasishtha
Abhinav Lamba
Anant Dalmia
Drush Subramanian
Anant Khare
Vikas Mangal
Vishal Bajaj
Avani Parikh
Mahima Grekar


In [116]:
#Evaluation
clean_names = [name[1:-1] for name in names]

generated = [generate(model) for _ in range(200)]

novel = [n for n in generated if n not in clean_names]
novelty = len(novel) / len(generated)

diversity = len(set(generated)) / len(generated)

print("\nEvaluation:")
print("Novelty:", novelty)
print("Diversity:", diversity)


Evaluation:
Novelty: 0.38
Diversity: 0.79


In [117]:
print_model_info(model, "RNN + Attention", lr=0.0015, epochs=60)


Model: RNN + Attention
Trainable Parameters - 211251
Embedding Dimension  - 128
Hidden Size          - 128
Number of Layers     - 1
Dropout              - 0.3
Learning Rate        - 0.0015
Epochs               - 60
